In [1]:
import time

import pandas as pd

from src.data.load_raw import load_raw_data
from src.data.preprocess import (
    extract_xy,
    merge_features_and_labels,
    standardize_splits,
)
from src.data.split import split_labeled_transactions
from src.evaluation.metrics import calculate_binary_metrics
from src.evaluation.threshold import find_best_f1_threshold
from src.models.classic import (
    build_dummy_classifier,
    build_logistic_regression,
    build_random_forest,
    build_xgboost,
)

In [2]:
features_df, classes_df, edges_df = load_raw_data()

transactions_df = merge_features_and_labels(
    features_df,
    classes_df,
)

splits = split_labeled_transactions(transactions_df)

print("Features:", features_df.shape)
print("Classes:", classes_df.shape)
print("Edges:", edges_df.shape)

print("\nTrain:", len(splits.train))
print("Validation:", len(splits.validation))
print("Test:", len(splits.test))

print("\nTrain labels:")
print(splits.train["target"].value_counts())

print("\nValidation labels:")
print(splits.validation["target"].value_counts())

print("\nTest labels:")
print(splits.test["target"].value_counts())

Features: (203769, 167)
Classes: (203769, 2)
Edges: (234355, 2)

Train: 29894
Validation: 7829
Test: 8841

Train labels:
target
0    26432
1     3462
Name: count, dtype: Int64

Validation labels:
target
0    7154
1     675
Name: count, dtype: Int64

Test labels:
target
0    8433
1     408
Name: count, dtype: Int64


In [3]:
x_train, y_train = extract_xy(
    splits.train,
    feature_set="local",
)

x_validation, y_validation = extract_xy(
    splits.validation,
    feature_set="local",
)

x_test, y_test = extract_xy(
    splits.test,
    feature_set="local",
)

(
    scaler,
    x_train_scaled,
    x_validation_scaled,
    x_test_scaled,
) = standardize_splits(
    x_train,
    x_validation,
    x_test,
)

In [4]:
results = []


def run_experiment(
    model_name,
    model,
    x_train,
    y_train,
    x_validation,
    y_validation,
    x_test,
    y_test,
):
    start_time = time.perf_counter()

    model.fit(
        x_train,
        y_train,
    )

    training_time = time.perf_counter() - start_time

    validation_scores = model.predict_proba(
        x_validation,
    )[:, 1]

    threshold = find_best_f1_threshold(
        y_validation,
        validation_scores,
    )

    test_scores = model.predict_proba(
        x_test,
    )[:, 1]

    metrics = calculate_binary_metrics(
        y_test,
        test_scores,
        threshold=threshold,
    )

    metrics.update(
        {
            "model": model_name,
            "feature_set": "local",
            "training_time_seconds": training_time,
        }
    )

    results.append(metrics)

    return model, metrics

In [5]:
dummy_model, dummy_metrics = run_experiment(
    model_name="Dummy Classifier",
    model=build_dummy_classifier(),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test,
)

In [6]:
logistic_model, logistic_metrics = run_experiment(
    model_name="Logistic Regression",
    model=build_logistic_regression(),
    x_train=x_train_scaled,
    y_train=y_train,
    x_validation=x_validation_scaled,
    y_validation=y_validation,
    x_test=x_test_scaled,
    y_test=y_test,
)


In [7]:
random_forest_model, random_forest_metrics = run_experiment(
    model_name="Random Forest",
    model=build_random_forest(),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test,
)

In [8]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", scale_pos_weight)

xgboost_model, xgboost_metrics = run_experiment(
    model_name="XGBoost",
    model=build_xgboost(
        scale_pos_weight=scale_pos_weight,
    ),
    x_train=x_train,
    y_train=y_train,
    x_validation=x_validation,
    y_validation=y_validation,
    x_test=x_test,
    y_test=y_test,
)

scale_pos_weight: 7.634893125361063


In [9]:
def run_feature_set_experiment(
    feature_set,
    model_name,
    model,
):
    x_train, y_train = extract_xy(
        splits.train,
        feature_set=feature_set,
    )

    x_validation, y_validation = extract_xy(
        splits.validation,
        feature_set=feature_set,
    )

    x_test, y_test = extract_xy(
        splits.test,
        feature_set=feature_set,
    )

    start_time = time.perf_counter()

    model.fit(x_train, y_train)

    training_time = time.perf_counter() - start_time

    validation_scores = model.predict_proba(
        x_validation,
    )[:, 1]

    threshold = find_best_f1_threshold(
        y_validation,
        validation_scores,
    )

    test_scores = model.predict_proba(
        x_test,
    )[:, 1]

    metrics = calculate_binary_metrics(
        y_test,
        test_scores,
        threshold=threshold,
    )

    metrics.update(
        {
            "model": model_name,
            "feature_set": feature_set,
            "training_time_seconds": training_time,
        }
    )

    results.append(metrics)

    return model, metrics

In [10]:
rf_all_model, rf_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="Random Forest",
    model=build_random_forest(),
)

xgb_all_model, xgb_all_metrics = run_feature_set_experiment(
    feature_set="all",
    model_name="XGBoost",
    model=build_xgboost(
        scale_pos_weight=scale_pos_weight,
    ),
)

In [11]:
results_df = pd.DataFrame(results)

columns = [
    "model",
    "feature_set",
    "pr_auc",
    "roc_auc",
    "precision_illicit",
    "recall_illicit",
    "f1_illicit",
    "threshold",
    "training_time_seconds",
]

results_df[columns].sort_values(
    by="pr_auc",
    ascending=False,
).style.format(
    {
        "pr_auc": "{:.4f}",
        "roc_auc": "{:.4f}",
        "precision_illicit": "{:.4f}",
        "recall_illicit": "{:.4f}",
        "f1_illicit": "{:.4f}",
        "threshold": "{:.2f}",
        "training_time_seconds": "{:.2f}",
    }
)

display(results_df)

,pr_auc,roc_auc,precision_illicit,recall_illicit,f1_illicit,confusion_matrix,threshold,model,feature_set,training_time_seconds
0,0.046149,0.500000,0.046149,1.000000,0.088226,"[[0, 8433], [0, 408]]",0.01,Dummy Classifier,local,0.000909
1,0.131483,0.798537,0.172332,0.534314,0.260610,"[[7386, 1047], [190, 218]]",0.93,Logistic Regression,local,0.215325
2,0.532786,0.805511,0.865385,0.441176,0.584416,"[[8405, 28], [228, 180]]",0.51,Random Forest,local,1.339945
3,0.532905,0.817730,0.774892,0.438725,0.560250,"[[8381, 52], [229, 179]]",0.78,XGBoost,local,1.265653
4,0.545015,0.850533,0.945274,0.465686,0.623974,"[[8422, 11], [218, 190]]",0.44,Random Forest,all,1.643929
5,0.555532,0.855694,0.940887,0.468137,0.625205,"[[8421, 12], [217, 191]]",0.91,XGBoost,all,1.501431
